# Engenharia de Features
## Construção do Dataset para Modelagem

**Objetivo:** Transformar os dados brutos de resultados de partidas em um dataset estruturado com features e target por seleção e por Copa do Mundo.

**Lógica central:**
- Para cada Copa (1994–2022), calculamos as features baseadas no **ciclo preparatório** — jogos disputados entre o fim da Copa anterior e o início da Copa alvo, excluindo jogos de Copa do Mundo
- O **target** é a média de gols marcados por jogo dentro da própria Copa

**Dataset de entrada:** `data/raw/results.csv`  
**Dataset de saída:** `data/processed/features_completo.csv`

## 1. Imports e Carregamento dos Dados

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/results.csv', parse_dates=['date'])

print(f'Shape: {df.shape}')
df.head()

Shape: (49287, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


## 2. Explorando a Copa 2022 (Exemplo)

Antes de construir o pipeline completo, vamos entender a estrutura dos dados para uma Copa específica.

Primeiro, identificamos as seleções participantes pegando mandantes **e** visitantes — apenas olhar para `home_team` não basta, pois nem toda seleção joga como mandante.

In [2]:
# Filtrar jogos da Copa 2022
copa_2022 = df[
    (df['tournament'] == 'FIFA World Cup') &
    (df['date'].dt.year == 2022)
]

# Pegar todas as seleções (mandante + visitante) sem repetição
selecoes_2022 = pd.unique(
    copa_2022[['home_team', 'away_team']].values.ravel()
)

print(f'Total de seleções na Copa 2022: {len(selecoes_2022)}')
print(selecoes_2022)

Total de seleções na Copa 2022: 32
['Qatar' 'Ecuador' 'Senegal' 'Netherlands' 'England' 'Iran'
 'United States' 'Wales' 'Argentina' 'Saudi Arabia' 'Mexico' 'Poland'
 'Denmark' 'Tunisia' 'France' 'Australia' 'Germany' 'Japan' 'Spain'
 'Costa Rica' 'Morocco' 'Croatia' 'Belgium' 'Canada' 'Switzerland'
 'Cameroon' 'Brazil' 'Serbia' 'Uruguay' 'South Korea' 'Portugal' 'Ghana']


## 3. Definindo o Ciclo Preparatório (Exemplo: Copa 2022)

O ciclo começa **após** o encerramento da Copa anterior e termina **antes** do início da Copa alvo.

- Ciclo 2022: `16/07/2018` (dia após a final da Copa 2018) → `19/11/2022` (dia anterior ao início da Copa 2022)
- Jogos de Copa do Mundo são **excluídos** do ciclo — eles são usados apenas como target

In [3]:
ciclo_inicio = '2018-07-16'  # dia após a final da Copa 2018
ciclo_fim = '2022-11-19'     # dia anterior ao início da Copa 2022

ciclo_2022 = df[
    (df['date'] >= ciclo_inicio) &
    (df['date'] <= ciclo_fim) &
    (df['tournament'] != 'FIFA World Cup')  # excluir jogos de Copa do Mundo
]

print(f'Total de jogos no ciclo: {len(ciclo_2022)}')
print(f'Período: {ciclo_2022["date"].min().date()} até {ciclo_2022["date"].max().date()}')

Total de jogos no ciclo: 3993
Período: 2018-07-22 até 2022-11-19


## 4. Calculando Features para uma Seleção (Exemplo: Brasil)

Para entender o que a função vai fazer, calculamos manualmente para o Brasil primeiro.

### 4.1 Jogos do Brasil no Ciclo

In [4]:
selecao = 'Brazil'

# Filtrar jogos do Brasil no ciclo (como mandante ou visitante)
jogos_brasil = ciclo_2022[
    (ciclo_2022['home_team'] == selecao) |
    (ciclo_2022['away_team'] == selecao)
].sort_values('date')

print(f'Total de jogos do Brasil no ciclo 2022: {len(jogos_brasil)}')
jogos_brasil[['date', 'home_team', 'away_team', 'home_score', 'away_score']].head(10)

Total de jogos do Brasil no ciclo 2022: 50


,date,home_team,away_team,home_score,away_score
41735,2018-09-07,United States,Brazil,0.0,2.0
41835,2018-09-12,Brazil,El Salvador,5.0,0.0
41877,2018-10-12,Saudi Arabia,Brazil,0.0,2.0
41985,2018-10-16,Argentina,Brazil,0.0,1.0
42042,2018-11-16,Brazil,Uruguay,1.0,0.0
42133,2018-11-20,Brazil,Cameroon,1.0,0.0
42304,2019-03-23,Brazil,Panama,1.0,1.0
42372,2019-03-26,Czech Republic,Brazil,1.0,3.0
42442,2019-06-05,Brazil,Qatar,2.0,0.0
42509,2019-06-09,Brazil,Honduras,7.0,0.0


### 4.2 Features do Ciclo Completo

Para cada jogo, extraímos os gols marcados e sofridos **do ponto de vista da seleção** — independente de ser mandante ou visitante.

In [5]:
gols_marcados = []
gols_sofridos = []
vitorias = []

for _, row in jogos_brasil.iterrows():
    if row['home_team'] == selecao:
        gols_marcados.append(row['home_score'])
        gols_sofridos.append(row['away_score'])
        vitorias.append(1 if row['home_score'] > row['away_score'] else 0)
    else:
        gols_marcados.append(row['away_score'])
        gols_sofridos.append(row['home_score'])
        vitorias.append(1 if row['away_score'] > row['home_score'] else 0)

print(f'Média de gols marcados no ciclo: {sum(gols_marcados)/len(gols_marcados):.2f}')
print(f'Média de gols sofridos no ciclo: {sum(gols_sofridos)/len(gols_sofridos):.2f}')
print(f'% de vitórias no ciclo:          {sum(vitorias)/len(vitorias):.2f}')
print(f'Total de jogos no ciclo:         {len(jogos_brasil)}')

Média de gols marcados no ciclo: 2.22
Média de gols sofridos no ciclo: 0.38
% de vitórias no ciclo:          0.74
Total de jogos no ciclo:         50


### 4.3 Features dos Últimos 15 Jogos

A janela de forma recente captura o momento atual da seleção — os 15 jogos mais recentes dentro do ciclo.

**Por que 15?** Representa aproximadamente 1,5 ano de jogos, equilibrando estabilidade estatística e recência. Janelas menores (10) são instáveis demais; janelas maiores (20) duplicam o que o ciclo já cobre.

In [6]:
# .tail(15) pega os 15 jogos mais recentes (o DataFrame já está ordenado por data)
ultimos_15 = jogos_brasil.tail(15)

gols_marcados_15 = []
gols_sofridos_15 = []
vitorias_15 = []

for _, row in ultimos_15.iterrows():
    if row['home_team'] == selecao:
        gols_marcados_15.append(row['home_score'])
        gols_sofridos_15.append(row['away_score'])
        vitorias_15.append(1 if row['home_score'] > row['away_score'] else 0)
    else:
        gols_marcados_15.append(row['away_score'])
        gols_sofridos_15.append(row['home_score'])
        vitorias_15.append(1 if row['away_score'] > row['home_score'] else 0)

print(f'Média de gols marcados (últ. 15): {sum(gols_marcados_15)/len(gols_marcados_15):.2f}')
print(f'Média de gols sofridos (últ. 15): {sum(gols_sofridos_15)/len(gols_sofridos_15):.2f}')
print(f'% de vitórias (últ. 15):          {sum(vitorias_15)/len(vitorias_15):.2f}')

Média de gols marcados (últ. 15): 2.53
Média de gols sofridos (últ. 15): 0.33
% de vitórias (últ. 15):          0.80


### 4.4 Target — Média de Gols na Copa

O target é calculado com os jogos **dentro** da Copa 2022. Esses dados nunca entram como features — apenas como variável alvo.

In [7]:
brasil_copa = copa_2022[
    (copa_2022['home_team'] == selecao) |
    (copa_2022['away_team'] == selecao)
]

gols_copa = []
for _, row in brasil_copa.iterrows():
    if row['home_team'] == selecao:
        gols_copa.append(row['home_score'])
    else:
        gols_copa.append(row['away_score'])

print(f'Jogos do Brasil na Copa 2022: {len(gols_copa)}')
print(f'Gols por jogo: {gols_copa}')
print(f'Média de gols na Copa 2022 (target): {sum(gols_copa)/len(gols_copa):.2f}')

Jogos do Brasil na Copa 2022: 5
Gols por jogo: [2.0, 1.0, 0.0, 4.0, 1.0]
Média de gols na Copa 2022 (target): 1.60


## 5. Função `calcular_features`

Agora encapsulamos toda a lógica acima em uma função reutilizável. Ela recebe:
- `selecao` — nome da seleção
- `ciclo` — DataFrame com os jogos do ciclo preparatório
- `copa` — DataFrame com os jogos da Copa alvo

E retorna um dicionário com todas as features e o target.

In [8]:
def calcular_features(selecao, ciclo, copa):
    # --- Jogos do ciclo ---
    jogos_selecao = ciclo[
        (ciclo['home_team'] == selecao) |
        (ciclo['away_team'] == selecao)
    ].sort_values('date')

    # Features do ciclo completo
    gols_marcados = []
    gols_sofridos = []
    vitorias = []

    for _, row in jogos_selecao.iterrows():
        if row['home_team'] == selecao:
            gols_marcados.append(row['home_score'])
            gols_sofridos.append(row['away_score'])
            vitorias.append(1 if row['home_score'] > row['away_score'] else 0)
        else:
            gols_marcados.append(row['away_score'])
            gols_sofridos.append(row['home_score'])
            vitorias.append(1 if row['away_score'] > row['home_score'] else 0)

    # Features dos últimos 15 jogos do ciclo
    ultimos_15 = jogos_selecao.tail(15)

    gols_marcados_15 = []
    gols_sofridos_15 = []
    vitorias_15 = []

    for _, row in ultimos_15.iterrows():
        if row['home_team'] == selecao:
            gols_marcados_15.append(row['home_score'])
            gols_sofridos_15.append(row['away_score'])
            vitorias_15.append(1 if row['home_score'] > row['away_score'] else 0)
        else:
            gols_marcados_15.append(row['away_score'])
            gols_sofridos_15.append(row['home_score'])
            vitorias_15.append(1 if row['away_score'] > row['home_score'] else 0)

    # --- Target: média de gols na Copa ---
    selecao_copa = copa[
        (copa['home_team'] == selecao) |
        (copa['away_team'] == selecao)
    ]

    gols_copa = []
    for _, row in selecao_copa.iterrows():
        if row['home_team'] == selecao:
            gols_copa.append(row['home_score'])
        else:
            gols_copa.append(row['away_score'])

    return {
        'media_gols_marcados_ciclo': sum(gols_marcados) / len(gols_marcados),
        'media_gols_sofridos_ciclo': sum(gols_sofridos) / len(gols_sofridos),
        'pct_vitorias_ciclo':        sum(vitorias) / len(vitorias),
        'total_jogos_ciclo':         len(jogos_selecao),
        'media_gols_marcados_ult15': sum(gols_marcados_15) / len(gols_marcados_15),
        'media_gols_sofridos_ult15': sum(gols_sofridos_15) / len(gols_sofridos_15),
        'pct_vitorias_ult15':        sum(vitorias_15) / len(vitorias_15),
        'media_gols_copa':           sum(gols_copa) / len(gols_copa)
    }


# Teste com o Brasil — valores devem bater com os calculados manualmente
resultado = calcular_features('Brazil', ciclo_2022, copa_2022)
print('Teste com o Brasil:')
for k, v in resultado.items():
    print(f'  {k}: {v:.4f}')

Teste com o Brasil:
  media_gols_marcados_ciclo: 2.2200
  media_gols_sofridos_ciclo: 0.3800
  pct_vitorias_ciclo: 0.7400
  total_jogos_ciclo: 50.0000
  media_gols_marcados_ult15: 2.5333
  media_gols_sofridos_ult15: 0.3333
  pct_vitorias_ult15: 0.8000
  media_gols_copa: 1.6000


## 6. Pipeline Completo — Todas as Copas (1994–2022)

Agora aplicamos a função para todas as Copas e todas as seleções participantes.

### 6.1 Dicionário de Datas dos Ciclos

Cada Copa tem seu próprio ciclo:
- **Início do ciclo:** dia após a final da Copa anterior
- **Fim do ciclo:** dia anterior ao início da Copa alvo

In [9]:
copas = {
    1994: {'ciclo_inicio': '1990-07-09', 'ciclo_fim': '1994-06-16', 'copa_inicio': '1994-06-17', 'copa_fim': '1994-07-17'},
    1998: {'ciclo_inicio': '1994-07-18', 'ciclo_fim': '1998-06-09', 'copa_inicio': '1998-06-10', 'copa_fim': '1998-07-12'},
    2002: {'ciclo_inicio': '1998-07-13', 'ciclo_fim': '2002-05-30', 'copa_inicio': '2002-05-31', 'copa_fim': '2002-06-30'},
    2006: {'ciclo_inicio': '2002-07-01', 'ciclo_fim': '2006-06-08', 'copa_inicio': '2006-06-09', 'copa_fim': '2006-07-09'},
    2010: {'ciclo_inicio': '2006-07-10', 'ciclo_fim': '2010-06-10', 'copa_inicio': '2010-06-11', 'copa_fim': '2010-07-11'},
    2014: {'ciclo_inicio': '2010-07-12', 'ciclo_fim': '2014-06-11', 'copa_inicio': '2014-06-12', 'copa_fim': '2014-07-13'},
    2018: {'ciclo_inicio': '2014-07-14', 'ciclo_fim': '2018-06-13', 'copa_inicio': '2018-06-14', 'copa_fim': '2018-07-15'},
    2022: {'ciclo_inicio': '2018-07-16', 'ciclo_fim': '2022-11-19', 'copa_inicio': '2022-11-20', 'copa_fim': '2022-12-18'},
}

print('Ciclos definidos para as Copas:')
for ano, datas in copas.items():
    print(f'  {ano}: {datas["ciclo_inicio"]} → {datas["ciclo_fim"]}')

Ciclos definidos para as Copas:
  1994: 1990-07-09 → 1994-06-16
  1998: 1994-07-18 → 1998-06-09
  2002: 1998-07-13 → 2002-05-30
  2006: 2002-07-01 → 2006-06-08
  2010: 2006-07-10 → 2010-06-10
  2014: 2010-07-12 → 2014-06-11
  2018: 2014-07-14 → 2018-06-13
  2022: 2018-07-16 → 2022-11-19


### 6.2 Loop Principal

Para cada Copa:
1. Filtra os jogos do ciclo preparatório (excluindo Copas do Mundo)
2. Identifica as seleções participantes
3. Calcula as features para cada seleção
4. Empilha os resultados

O `try/except` protege contra seleções com poucos dados no ciclo que causariam divisão por zero.

In [10]:
todos_dados = []

for ano, datas in copas.items():
    print(f'Processando Copa {ano}...')

    # Jogos do ciclo preparatório (sem Copa do Mundo)
    ciclo = df[
        (df['date'] >= datas['ciclo_inicio']) &
        (df['date'] <= datas['ciclo_fim']) &
        (df['tournament'] != 'FIFA World Cup')
    ]

    # Jogos dentro da Copa alvo (usados apenas para o target)
    copa = df[
        (df['tournament'] == 'FIFA World Cup') &
        (df['date'] >= datas['copa_inicio']) &
        (df['date'] <= datas['copa_fim'])
    ]

    # Seleções participantes da Copa
    selecoes = pd.unique(copa[['home_team', 'away_team']].values.ravel())

    for selecao in selecoes:
        try:
            resultado = calcular_features(selecao, ciclo, copa)
            resultado['selecao'] = selecao
            resultado['copa_alvo'] = ano
            todos_dados.append(resultado)
        except Exception as e:
            print(f'  Erro em {selecao}: {e}')

df_final = pd.DataFrame(todos_dados)
print(f'\nDataset final: {df_final.shape[0]} linhas × {df_final.shape[1]} colunas')
df_final.head()

Processando Copa 1994...
Processando Copa 1998...
Processando Copa 2002...
Processando Copa 2006...
Processando Copa 2010...
Processando Copa 2014...
Processando Copa 2018...
Processando Copa 2022...

Dataset final: 248 linhas × 10 colunas


,media_gols_marcados_ciclo,media_gols_sofridos_ciclo,pct_vitorias_ciclo,total_jogos_ciclo,media_gols_marcados_ult15,media_gols_sofridos_ult15,pct_vitorias_ult15,media_gols_copa,selecao,copa_alvo
0,1.975610,0.975610,0.609756,41,2.533333,1.066667,0.666667,1.800000,Germany,1994
1,1.023810,1.190476,0.214286,42,0.600000,1.133333,0.133333,0.333333,Bolivia,1994
2,1.885714,0.914286,0.514286,35,2.266667,0.733333,0.666667,2.000000,Spain,1994
3,1.750000,0.625000,0.517857,56,1.533333,1.066667,0.333333,1.333333,South Korea,1994
4,1.279070,0.604651,0.465116,43,1.733333,0.400000,0.600000,1.333333,Colombia,1994


## 7. Validação do Dataset

Verificamos se o número de linhas por Copa está correto:
- 1994: 24 seleções (formato antigo)
- 1998–2022: 32 seleções

In [11]:
print(f'Total de linhas: {df_final.shape[0]}')
print(f'Copas incluídas: {sorted(df_final["copa_alvo"].unique())}')
print(f'\nLinhas por Copa:')
print(df_final['copa_alvo'].value_counts().sort_index())

print(f'\nEstatísticas do target (media_gols_copa):')
print(df_final['media_gols_copa'].describe().round(2))

Total de linhas: 248
Copas incluídas: [1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]

Linhas por Copa:
copa_alvo
1994    24
1998    32
2002    32
2006    32
2010    32
2014    32
2018    32
2022    32
Name: count, dtype: int64

Estatísticas do target (media_gols_copa):
count    248.00
mean       1.17
std        0.59
min        0.00
25%        0.67
50%        1.14
75%        1.57
max        2.67
Name: media_gols_copa, dtype: float64


## 8. Salvando o Dataset Processado

O arquivo é salvo em `data/processed/` — separado dos dados brutos conforme exigido pelo RQ-ER-02.

In [12]:
df_final.to_csv('../data/processed/features_completo.csv', index=False)
print('Dataset salvo em: data/processed/features_completo.csv')
print(f'Shape final: {df_final.shape}')

Dataset salvo em: data/processed/features_completo.csv
Shape final: (248, 10)


## 9. Resumo

**Features construídas por seleção + Copa:**

| Feature | Descrição |
|---------|----------|
| `media_gols_marcados_ciclo` | Média de gols marcados no ciclo completo |
| `media_gols_sofridos_ciclo` | Média de gols sofridos no ciclo completo |
| `pct_vitorias_ciclo` | % de vitórias no ciclo completo |
| `total_jogos_ciclo` | Total de jogos disputados no ciclo |
| `media_gols_marcados_ult15` | Média de gols marcados nos últimos 15 jogos |
| `media_gols_sofridos_ult15` | Média de gols sofridos nos últimos 15 jogos |
| `pct_vitorias_ult15` | % de vitórias nos últimos 15 jogos |
| `media_gols_copa` | **TARGET** — média de gols marcados na Copa |

**Próximo passo:** `03_modelos.ipynb` — treino e avaliação dos modelos de regressão.